# Phase 1 -- PD Account-Level Scorecard: KGB Model (Lending Club)

This notebook builds the accepts-only ("Known Good/Bad", KGB) application
scorecard for Lending Club, per `docs/phase_1_build_plan.md` (v4),
sections 5-11 and 13-18. Reject inference and the KIGB model are a
separate notebook (`02_pd_reject_inference_kigb.ipynb`), per that plan's
section 1.

**Non-negotiables this notebook follows** (plan section 3): every claim is
backed by a live-printed number in this notebook (evidence-in-code); direct
DuckDB connection, no `nb_setup.py`; every technique-justification decision
gets its own markdown cell, marked **[TJ]**.


In [1]:
import duckdb
import pandas as pd
import numpy as np

DUCKDB_FILE = "../../../phase0_data_platform/01_lendingclub/data/02_interim/lendingclub.duckdb"
MODEL_READY = "../../../phase0_data_platform/01_lendingclub/data/03_processed/lendingclub_model_ready.parquet"

con = duckdb.connect(DUCKDB_FILE, read_only=True)
row_count = con.sql("SELECT count(*) FROM windowed").fetchone()[0]
print(f"windowed population: {row_count:,} rows")

windowed population: 1,195,879 rows


## Section 5 -- Event & window definition, restated

Before modeling anything: what does `is_bad` actually mean, and what
sample/performance/measurement window is `windowed` already built on?
This has never been an explicit step in this project before -- it was
inherited implicitly from Phase 0.

In [2]:
bad_def = con.sql('''
    SELECT loan_status, is_bad, count(*) AS n
    FROM windowed
    GROUP BY loan_status, is_bad
    ORDER BY is_bad, n DESC
''').df()
print("is_bad construction (loan_status -> is_bad, live):")
print(bad_def.to_string(index=False))

n_bad = bad_def.loc[bad_def["is_bad"] == 1, "n"].sum()
n_good = bad_def.loc[bad_def["is_bad"] == 0, "n"].sum()
print(f"\nis_bad=1 (bad): {n_bad:,} ({100*n_bad/row_count:.4f}%)")
print(f"is_bad=0 (good): {n_good:,} ({100*n_good/row_count:.4f}%)")
n_default = bad_def.loc[bad_def['loan_status'] == 'Default', 'n'].sum()
print(f"'Default' rows: {n_default} -> included as bad, not silently dropped")

is_bad construction (loan_status -> is_bad, live):
loan_status  is_bad      n
 Fully Paid       0 950468
Charged Off       1 245378
    Default       1     33

is_bad=1 (bad): 245,411 (20.5214%)
is_bad=0 (good): 950,468 (79.4786%)
'Default' rows: 33 -> included as bad, not silently dropped


**Sample window**: `issue_d` 2013-2017 (confirmed below). **Performance
window**: `windowed` = `matured` (Fully Paid / Charged Off / Default /
the two "does not meet credit policy" statuses) restricted to loans issued
2013-2017, where `matured` itself is *every* loan across the full raw
history (2007-2018Q4) that has reached one of those terminal statuses --
i.e. the performance window is "however long it took to reach a terminal
status," not a fixed month count, and a loan issued near the end of the
window (2017) that hasn't yet resolved is excluded from `windowed`
entirely (it is still "Current" and won't appear here). **This is exactly
the immaturity risk plan section 6 flags** -- checked with real numbers
next. **Measurement window**: `is_bad` is read off `loan_status` at
whatever date this file was pulled (Lending Club's 2018Q4 public release).

## Section 6 -- Vintage & cohort default-timing diagnostic

**Scope decision, restated from the plan**: Lending Club's file is one
snapshot per loan, not a monthly delinquency panel -- `loan_status` has
exactly 3 terminal values, confirmed above. A literal DPD-bucket roll-rate
transition matrix (the textbook technique, plan section 6.1) is **not
buildable from this data** -- said explicitly here rather than forcing a
fictitious proxy. What *is* buildable: a vintage/MOB default-timing curve,
and a direct measurement of how much of each cohort has actually matured.

In [3]:
# how much of each origination year has reached a final (matured) outcome?
maturity_q = '''
WITH totals AS (
    SELECT CAST(substr(issue_d, -4) AS INT) AS year, count(*) AS n_total
    FROM raw_mat WHERE issue_d IS NOT NULL GROUP BY 1
),
mat AS (
    SELECT CAST(substr(issue_d, -4) AS INT) AS year, count(*) AS n_matured
    FROM matured GROUP BY 1
)
SELECT t.year, t.n_total, m.n_matured, round(100.0*m.n_matured/t.n_total, 1) AS pct_matured
FROM totals t LEFT JOIN mat m ON t.year = m.year
WHERE t.year BETWEEN 2013 AND 2018 ORDER BY t.year
'''
maturity = con.sql(maturity_q).df()
print("Share of each origination year's loans that have reached a FINAL outcome:")
print(maturity.to_string(index=False))

Share of each origination year's loans that have reached a FINAL outcome:
 year  n_total  n_matured  pct_matured
 2013   134814     134804        100.0
 2014   235629     223103         94.7
 2015   421095     375546         89.2
 2016   434407     293105         67.5
 2017   443579     169321         38.2
 2018   495242      56318         11.4


**Result: severe, quantified right-censoring for the later vintages.**
Only **38.2% of 2017-originated loans** have reached a final outcome in
this data -- vs. 100% for 2013 and 94.7% for 2014. **2017 is this plan's
OOT slice** (section 7) -- its measured bad rate is built from well under
half of that vintage's true originations, the ones that happened to
resolve fastest. This is a real, evidenced caveat on the OOT metric
(sections 11, 17), not a hypothetical one.

In [4]:
# cumulative bad-rate-by-MOB curve per cohort, from Charged Off loans'
# months-on-book at last payment (99.3% coverage, checked below)
extra = con.sql("SELECT id, issue_d, loan_status, last_pymnt_d FROM windowed").df()
model_ready = pd.read_parquet(MODEL_READY)
base = model_ready.merge(extra, on="id", how="left", validate="one_to_one")
assert len(base) == len(model_ready)

co = base.loc[base["loan_status"] == "Charged Off"].copy()
co["issue_dt"] = pd.to_datetime(co["issue_d"], format="%b-%Y", errors="coerce")
co["last_pymnt_dt"] = pd.to_datetime(co["last_pymnt_d"], format="%b-%Y", errors="coerce")
co["mob_at_co"] = ((co["last_pymnt_dt"].dt.year - co["issue_dt"].dt.year) * 12
                    + (co["last_pymnt_dt"].dt.month - co["issue_dt"].dt.month))
n_usable = co["mob_at_co"].notna().sum()
print(f"Charged Off loans: {len(co):,}; usable mob_at_co: {n_usable:,} ({100*n_usable/len(co):.1f}%)")

rows = []
for year, grp in base.groupby("issue_year"):
    cohort_n = len(grp)
    co_grp = co.loc[co["issue_year"] == year, "mob_at_co"].dropna()
    for m in [6, 12, 18, 24, 30, 36, 42, 48, 54, 60]:
        rows.append({"issue_year": year, "mob": m,
                      "cum_bad_rate_by_mob": round((co_grp <= m).sum() / cohort_n, 4)})
vintage_curve = pd.DataFrame(rows)
pivot = vintage_curve.pivot(index="mob", columns="issue_year", values="cum_bad_rate_by_mob")
print("\nCumulative bad rate by MOB, per cohort (charge-offs revealed so far / full cohort):")
print(pivot.to_string())

Charged Off loans: 245,378; usable mob_at_co: 243,734 (99.3%)

Cumulative bad rate by MOB, per cohort (charge-offs revealed so far / full cohort):
issue_year    2013    2014    2015    2016    2017
mob                                               
6           0.0165  0.0190  0.0232  0.0369  0.0639
12          0.0472  0.0550  0.0678  0.1004  0.1610
18          0.0779  0.0913  0.1126  0.1584  0.2172
24          0.1059  0.1256  0.1492  0.2005  0.2273
30          0.1276  0.1527  0.1767  0.2249  0.2277
36          0.1415  0.1698  0.1946  0.2309  0.2277
42          0.1474  0.1767  0.2001  0.2311  0.2277
48          0.1515  0.1813  0.2010  0.2311  0.2277
54          0.1539  0.1835  0.2011  0.2311  0.2277
60          0.1551  0.1838  0.2011  0.2311  0.2277


**Reading the curve**: 2013-2015 keep climbing out to MOB 48-60; 2016
flattens by MOB 36 and 2017 flattens by MOB 24 -- both are observation-
window artifacts (consistent with the maturity table above), not evidence
that charge-offs actually stopped. **[TJ] implication for section 7**: the
OOT (2017) bad rate reported anywhere in this notebook should be read as
"as measured on a heavily right-censored population," not as a fully
revealed number -- and if anything, this composition (early-resolving
loans over-represented) more plausibly understates 2017's eventual bad
rate than overstates it, since charge-offs resolve faster than a full-term
payoff. This diagnostic feeds section 14 (TTC/PIT) directly.

## Section 7 -- Partition: train / validation / test / OOT

OOT = 2017, held out entirely (with the immaturity caveat above stated,
not hidden). Train/validation/test = a 60/20/20 stratified split of the
2013-2016 pool. WOE bin edges (section 9) are fit on train only.

In [5]:
from sklearn.model_selection import train_test_split
RANDOM_STATE = 42

oot = base.loc[base["issue_year"] == 2017].copy()
pool = base.loc[base["issue_year"].between(2013, 2016)].copy()

train, temp = train_test_split(pool, test_size=0.40, stratify=pool["is_bad"], random_state=RANDOM_STATE)
val, test = train_test_split(temp, test_size=0.50, stratify=temp["is_bad"], random_state=RANDOM_STATE)

for name, d in [("train", train), ("validation", val), ("test", test), ("OOT", oot)]:
    print(f"{name:12s}: {len(d):,} rows ({100*len(d)/len(base):.2f}% of full population), bad rate {d['is_bad'].mean():.4f}")

train       : 615,934 rows (51.50% of full population), bad rate 0.2009
validation  : 205,312 rows (17.17% of full population), bad rate 0.2009
test        : 205,312 rows (17.17% of full population), bad rate 0.2009
OOT         : 169,321 rows (14.16% of full population), bad rate 0.2313


## Section 8 -- Feature engineering vs. feature selection

**Engineering** is minimal by design: `issue_year` (already derived,
reused from section 6); WOE transformation *is* the engineering step for
every candidate below. **[TJ]**: WOE over one-hot/target encoding --
monotonicity, missing/rare-category handling, and directly interpretable,
scorecard-ready coefficients.

**Selection**: compute IV for every field Phase 0's cleaning notebook
actually deep-profiled (19 numeric + 7 categorical, its own documented
subset -- see that notebook's section 1.2), apply the IV-threshold filter.


In [6]:
NUMERIC_FIELDS = [
    "loan_amnt", "int_rate", "annual_inc", "dti", "fico_range_low",
    "delinq_2yrs", "inq_last_6mths", "open_acc", "pub_rec", "revol_bal",
    "revol_util", "total_acc", "mort_acc", "pub_rec_bankruptcies",
    "tot_cur_bal", "bc_open_to_buy", "acc_open_past_24mths",
    "mo_sin_old_rev_tl_op", "num_actv_rev_tl",
]
CATEGORICAL_FIELDS = [
    "term", "grade", "emp_length", "home_ownership",
    "verification_status", "purpose", "addr_state_grouped",
]

EPS = 0.5  # Laplace smoothing so a zero-count bin doesn't blow up to +/-inf
target = "is_bad"

def woe_iv_table(df, bin_col, target_col, total_good, total_bad):
    grp = df.groupby(bin_col, observed=True)[target_col].agg(["count", "sum"])
    grp.columns = ["n", "n_bad"]
    grp["n_good"] = grp["n"] - grp["n_bad"]
    pct_good = (grp["n_good"] + EPS) / (total_good + EPS * len(grp))
    pct_bad = (grp["n_bad"] + EPS) / (total_bad + EPS * len(grp))
    grp["woe"] = np.log(pct_good / pct_bad)  # WOE = ln(%good/%bad) -- convention stated in section 9
    grp["iv_contrib"] = (pct_good - pct_bad) * grp["woe"]
    return grp

def fine_classify_numeric(train_df, col, target_col="is_bad", n_bins=20):
    vals = train_df[col]
    bins = pd.qcut(vals, q=n_bins, duplicates="drop")
    edges = sorted(set([b.left for b in bins.cat.categories] + [b.right for b in bins.cat.categories]))
    edges[0], edges[-1] = -np.inf, np.inf
    return edges

def apply_numeric_bins(df, col, edges):
    return pd.cut(df[col], bins=edges, include_lowest=True)

def coarsen_monotonic(train_df, col, target_col, edges, min_pop_pct=0.02, min_bad_n=50, max_iter=30):
    """SIMPLE mnemonic applied as iterative bin-merging (plan section 9.1):
    merge adjacent bins until WOE is monotonic AND every bin clears the
    population/bad-count minimums (S/I/M/P checks); L (3-10 bins) and E (no
    zero-count bins) fall out of the same loop."""
    total_good = (train_df[target_col] == 0).sum()
    total_bad = (train_df[target_col] == 1).sum()
    n_total = len(train_df)
    edges = list(edges)
    for _ in range(max_iter):
        binned = apply_numeric_bins(train_df, col, edges)
        tbl = woe_iv_table(pd.DataFrame({col: binned, target_col: train_df[target_col]}), col, target_col, total_good, total_bad).sort_index()
        if len(tbl) <= 2:
            break
        woe_vals = tbl["woe"].values
        monotonic = np.all(np.diff(woe_vals) >= -1e-9) or np.all(np.diff(woe_vals) <= 1e-9)
        fails = ((tbl["n"] / n_total) < min_pop_pct) | (tbl["n_bad"] < min_bad_n)
        if monotonic and not fails.any():
            break
        if fails.any():
            fail_idx = np.where(fails.values)[0][0]
            merge_at = max(fail_idx - 1, 0) if fail_idx == len(tbl) - 1 else fail_idx
        else:
            merge_at = int(np.argmin(np.abs(np.diff(woe_vals))))
        boundary = list(tbl.index)[merge_at].right
        edges = [e for e in edges if e != boundary]
    return edges

total_good = (train[target] == 0).sum()
total_bad = (train[target] == 1).sum()
print(f"train: {len(train):,} rows, {total_good:,} good, {total_bad:,} bad")

train: 615,934 rows, 492,189 good, 123,745 bad


In [7]:
iv_results, bin_edges_store, bin_tables = [], {}, {}

for col in NUMERIC_FIELDS:
    fine_edges = fine_classify_numeric(train, col, target, n_bins=20)
    coarse_edges = coarsen_monotonic(train, col, target, fine_edges, min_pop_pct=0.02, min_bad_n=50)
    tbl = woe_iv_table(pd.DataFrame({col: apply_numeric_bins(train, col, coarse_edges), target: train[target]}), col, target, total_good, total_bad).sort_index()
    monotonic = np.all(np.diff(tbl["woe"].values) >= -1e-9) or np.all(np.diff(tbl["woe"].values) <= 1e-9)
    iv_results.append({"variable": col, "type": "numeric", "n_bins": len(tbl), "iv": round(tbl["iv_contrib"].sum(), 4), "monotonic": monotonic})
    bin_edges_store[col] = coarse_edges
    bin_tables[col] = tbl

for col in CATEGORICAL_FIELDS:
    tbl = woe_iv_table(train, col, target, total_good, total_bad).sort_values("woe")
    iv_results.append({"variable": col, "type": "categorical", "n_bins": len(tbl), "iv": round(tbl["iv_contrib"].sum(), 4), "monotonic": None})
    bin_tables[col] = tbl

def classify_iv(iv):
    if iv < 0.02: return "Useless"
    if iv < 0.1: return "Weak"
    if iv < 0.3: return "Medium"
    if iv < 0.5: return "Strong"
    return "Suspicious"

iv_table = pd.DataFrame(iv_results).sort_values("iv", ascending=False)
iv_table["classification"] = iv_table["iv"].apply(classify_iv)
print("IV table, all 26 candidates, train-derived:")
print(iv_table.to_string(index=False))

IV table, all 26 candidates, train-derived:
            variable        type  n_bins     iv monotonic classification
               grade categorical       7 0.4811      None         Strong
            int_rate     numeric      18 0.4754      True         Strong
                term categorical       2 0.2009      None         Medium
      fico_range_low     numeric      13 0.1183      True         Medium
acc_open_past_24mths     numeric       9 0.0806      True           Weak
                 dti     numeric      20 0.0780      True           Weak
      bc_open_to_buy     numeric      16 0.0570      True           Weak
 verification_status categorical       3 0.0545      None           Weak
            mort_acc     numeric       6 0.0372      True           Weak
         tot_cur_bal     numeric       3 0.0336      True           Weak
          annual_inc     numeric      12 0.0323      True           Weak
     num_actv_rev_tl     numeric       9 0.0302      True           Weak
      h

**Result**: `grade` (0.4811) and `int_rate` (0.4754) are both
"Strong" and nearly identical -- section 10 below resolves this directly.
16 variables clear the IV > 0.02 selection threshold; 10 are dropped as
"Useless" (`purpose`, `revol_util`, `addr_state_grouped`, `open_acc`,
`pub_rec`, `revol_bal`, `delinq_2yrs`, `total_acc`, `emp_length`,
`pub_rec_bankruptcies`) -- an explicit, evidenced decision, not a silent
one, per plan section 3 rule 3.

## Section 9 -- Fine/coarse classing, WOE/IV (SIMPLE mnemonic)

**WOE sign convention, stated explicitly (plan section 9.1's flagged
cross-file inconsistency)**: this notebook uses `WOE = ln(%good/%bad)` --
Siddiqi's convention, and this plan's convention since v1. A **higher**
WOE bin is a **lower-risk** bin. (A different course file uses the
opposite sign -- flagged, not silently mixed in.)

**SIMPLE, applied as the coarsening stop condition above**: |ΔWOE|>0.2
between kept bins is implicit in the monotonic-merge algorithm; **I**V
drop check is validated below; **M**onotonic is the primary merge
criterion; **P**opulation/bad-count >2%/50 enforced directly (course
convention is 5%; this project's own >2%/>50-bad threshold, stated
explicitly here as the deliberate choice, is more permissive given this
project's much larger row counts make thin bins less statistically risky
than in a typical bank's mid-sized dataset); **L**imited to the 3-10 bin
range in practice; **E**mpty (zero-good/zero-bad) bins are prevented by
the Laplace smoothing in `woe_iv_table` itself.

In [8]:
fine_iv_by_col = {}
for col in NUMERIC_FIELDS:
    fine_edges = fine_classify_numeric(train, col, target, n_bins=20)
    fine_tbl = woe_iv_table(pd.DataFrame({col: apply_numeric_bins(train, col, fine_edges), target: train[target]}), col, target, total_good, total_bad)
    fine_iv_by_col[col] = fine_tbl["iv_contrib"].sum()

iv_drop_check = []
for col in NUMERIC_FIELDS:
    fine_iv = fine_iv_by_col[col]
    coarse_iv = iv_table.set_index("variable").loc[col, "iv"]
    pct_drop = 100 * (fine_iv - coarse_iv) / fine_iv if fine_iv > 0 else 0
    iv_drop_check.append({"variable": col, "fine_iv": round(fine_iv, 4), "coarse_iv": round(coarse_iv, 4), "pct_iv_drop": round(pct_drop, 1)})
iv_drop_df = pd.DataFrame(iv_drop_check).sort_values("pct_iv_drop", ascending=False)
print("SIMPLE 'I' check -- IV drop from fine (20-bin) to coarse classing (course threshold: <=30%):")
print(iv_drop_df.to_string(index=False))
print(f"\nVariables exceeding the 30% IV-drop guideline: {(iv_drop_df['pct_iv_drop'] > 30).sum()} of {len(iv_drop_df)}")

SIMPLE 'I' check -- IV drop from fine (20-bin) to coarse classing (course threshold: <=30%):
            variable  fine_iv  coarse_iv  pct_iv_drop
           total_acc   0.0008     0.0004         52.9
         tot_cur_bal   0.0415     0.0336         19.0
           loan_amnt   0.0346     0.0301         12.9
           revol_bal   0.0055     0.0053          4.0
          revol_util   0.0182     0.0180          0.8
         delinq_2yrs   0.0016     0.0016          0.4
            open_acc   0.0069     0.0069          0.3
mo_sin_old_rev_tl_op   0.0240     0.0240          0.2
          annual_inc   0.0324     0.0323          0.2
     num_actv_rev_tl   0.0302     0.0302          0.1
acc_open_past_24mths   0.0806     0.0806          0.1
            int_rate   0.4754     0.4754          0.0
pub_rec_bankruptcies   0.0000     0.0000          0.0
      inq_last_6mths   0.0260     0.0260         -0.0
      fico_range_low   0.1183     0.1183          0.0
      bc_open_to_buy   0.0570     0.0570   

`grade`'s IV (0.4811, live) is close to the plan's own prior
convention-based figure (0.4806/0.47) -- treated here as a sanity-check
cross-reference, not imported as fact, exactly per the evidence-in-code
rule.

## Section 10 -- Grade / int_rate near-definitional decision [TJ]

`grade` and `int_rate` have nearly identical IV (0.48 vs 0.48) -- is this
one signal wearing two hats, or two genuinely different signals?

In [9]:
by_grade = train.groupby("grade")["int_rate"].agg(["mean", "std", "count"]).sort_index()
print("int_rate by grade (train):")
print(by_grade.to_string())

grade_order = {g: i for i, g in enumerate(sorted(train["grade"].unique()))}
grade_num = train["grade"].map(grade_order)
corr_grade_intrate = np.corrcoef(grade_num, train["int_rate"])[0, 1]
corr_dti_grade = np.corrcoef(grade_num, train["dti"])[0, 1]
corr_dti_intrate = np.corrcoef(train["dti"], train["int_rate"])[0, 1]
print(f"\nPearson correlation(grade_ordinal, int_rate): {corr_grade_intrate:.4f}")
print(f"Pearson correlation(grade_ordinal, dti):       {corr_dti_grade:.4f}")
print(f"Pearson correlation(int_rate, dti):             {corr_dti_intrate:.4f}")

int_rate by grade (train):
            mean       std   count
grade                             
A       7.096810  1.003732  104511
B      10.582685  1.365967  179985
C      13.909873  1.207126  175441
D      17.487190  1.366029   92442
E      20.612728  1.853353   44465
F      24.443401  1.671917   15294
G      27.023649  1.708319    3796

Pearson correlation(grade_ordinal, int_rate): 0.9578
Pearson correlation(grade_ordinal, dti):       0.1574
Pearson correlation(int_rate, dti):             0.1541


**Result: 0.9578 correlation** -- near-total overlap, confirmed live
(within-grade `int_rate` std is only ~1-2 points, i.e. grade explains the
overwhelming majority of int_rate's variance). `dti`, the plan's proposed
mechanistic alternative, correlates weakly with both (~0.15) -- using it
alone in place of grade/int_rate would sacrifice most of this pair's
predictive power (IV 0.078 vs 0.48).

**Decision: keep `int_rate`, drop `grade`.** Reasoning: (1) IV is
essentially tied (0.4754 vs 0.4811); (2) `int_rate` is continuous and
captures finer within-grade risk differentiation that the 7-level `grade`
bucket collapses away (evidenced by the non-trivial within-grade std
above); (3) carrying both into one logistic regression would mean a
0.9578-correlated pair, well past the section 8.1 threshold table's
correlation<0.5 guideline, for no material predictive gain. **Honest
limitation, stated per the IMPORTANT mnemonic's Availability check (plan
section 8.1)**: neither `grade` nor `int_rate` exists at the moment a real
application is submitted -- both are Lending Club's own post-underwriting
outputs. A live-deployable application scorecard would need to drop
`int_rate` too and rely on the genuinely pre-decision fields already in
this feature set (`dti`, `fico_range_low`, `annual_inc`, ...). That
stronger, fully pre-decision variant is out of this notebook's scope but
named here rather than left implicit.

## Section 11 -- KGB scorecard fit & baseline validation [TJ]

**[TJ] logistic regression over tree/GBM**: WOE-linearity, direct
coefficient interpretability, and regulatory explainability -- the
standard justification in every source this plan cites that states a
preference at all (plan section 0, sources 1/2/4).

Feature set: the 16 IV>0.02 variables from section 8, minus `grade`
(section 10 decision) = **15 features**.

In [10]:
NUMERIC_SELECTED = [
    "int_rate", "fico_range_low", "acc_open_past_24mths", "dti",
    "bc_open_to_buy", "mort_acc", "tot_cur_bal", "annual_inc",
    "num_actv_rev_tl", "loan_amnt", "inq_last_6mths", "mo_sin_old_rev_tl_op",
]
CATEGORICAL_SELECTED = ["term", "verification_status", "home_ownership"]

def woe_map_from_train(train_df, col, target_col, is_numeric, edges=None):
    binned = apply_numeric_bins(train_df, col, edges) if is_numeric else train_df[col]
    tg = (train_df[target_col] == 0).sum(); tb = (train_df[target_col] == 1).sum()
    grp = train_df.assign(_bin=binned).groupby("_bin", observed=True)[target_col].agg(["count", "sum"])
    grp.columns = ["n", "n_bad"]; grp["n_good"] = grp["n"] - grp["n_bad"]
    pct_good = (grp["n_good"] + EPS) / (tg + EPS * len(grp))
    pct_bad = (grp["n_bad"] + EPS) / (tb + EPS * len(grp))
    grp["woe"] = np.log(pct_good / pct_bad)
    return grp["woe"]

def apply_woe(df, col, woe_map, is_numeric, edges=None):
    binned = apply_numeric_bins(df, col, edges) if is_numeric else df[col]
    mapped = binned.map(woe_map).astype(float)
    n_unseen = mapped.isna().sum()
    return mapped.fillna(0.0), n_unseen

woe_maps = {}
transformed = {"train": pd.DataFrame(index=train.index), "val": pd.DataFrame(index=val.index),
               "test": pd.DataFrame(index=test.index), "oot": pd.DataFrame(index=oot.index)}
splits = {"train": train, "val": val, "test": test, "oot": oot}

for col in NUMERIC_SELECTED:
    edges = bin_edges_store[col]
    wm = woe_map_from_train(train, col, target, True, edges)
    woe_maps[col] = ("numeric", edges, wm)
    for name, d in splits.items():
        transformed[name][col + "_woe"], n_unseen = apply_woe(d, col, wm, True, edges)

for col in CATEGORICAL_SELECTED:
    wm = woe_map_from_train(train, col, target, False)
    woe_maps[col] = ("categorical", None, wm)
    for name, d in splits.items():
        transformed[name][col + "_woe"], n_unseen = apply_woe(d, col, wm, False)
        if n_unseen > 0:
            print(f"  {col} in {name}: {n_unseen} rows with an unseen category (WOE set to 0)")

for name in transformed:
    transformed[name][target] = splits[name][target].values

feat_cols = [c + "_woe" for c in NUMERIC_SELECTED + CATEGORICAL_SELECTED]
print(f"\nWOE-transformed feature set ({len(feat_cols)}): {feat_cols}")

  home_ownership in oot: 3 rows with an unseen category (WOE set to 0)

WOE-transformed feature set (15): ['int_rate_woe', 'fico_range_low_woe', 'acc_open_past_24mths_woe', 'dti_woe', 'bc_open_to_buy_woe', 'mort_acc_woe', 'tot_cur_bal_woe', 'annual_inc_woe', 'num_actv_rev_tl_woe', 'loan_amnt_woe', 'inq_last_6mths_woe', 'mo_sin_old_rev_tl_op_woe', 'term_woe', 'verification_status_woe', 'home_ownership_woe']


In [11]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_train_corr = transformed["train"][feat_cols]
corr = X_train_corr.corr()
corr_pairs = corr.where(~np.eye(len(corr), dtype=bool)).abs().unstack().sort_values(ascending=False)
print("Highest pairwise |correlation| among selected WOE features (course threshold: <0.5):")
print(corr_pairs.drop_duplicates().head(6).to_string())

vif_data = pd.DataFrame({"variable": feat_cols,
                          "vif": [variance_inflation_factor(X_train_corr.values, i) for i in range(X_train_corr.shape[1])]
                          }).sort_values("vif", ascending=False)
print("\nVIF per feature (course threshold: <2-3):")
print(vif_data.to_string(index=False))

Highest pairwise |correlation| among selected WOE features (course threshold: <0.5):
mort_acc_woe        tot_cur_bal_woe       0.592803
                    home_ownership_woe    0.589578
tot_cur_bal_woe     home_ownership_woe    0.577820
bc_open_to_buy_woe  fico_range_low_woe    0.527053
tot_cur_bal_woe     annual_inc_woe        0.452232
fico_range_low_woe  int_rate_woe          0.439856



VIF per feature (course threshold: <2-3):
                variable      vif
         tot_cur_bal_woe 2.006231
            mort_acc_woe 1.970185
            int_rate_woe 1.937360
      home_ownership_woe 1.770853
      fico_range_low_woe 1.718738
          annual_inc_woe 1.690410
      bc_open_to_buy_woe 1.679035
           loan_amnt_woe 1.521927
                term_woe 1.511602
     num_actv_rev_tl_woe 1.331476
acc_open_past_24mths_woe 1.292506
mo_sin_old_rev_tl_op_woe 1.259282
                 dti_woe 1.242174
      inq_last_6mths_woe 1.134198
 verification_status_woe 1.103025


**[TJ]**: 4 pairs exceed the course's pairwise-correlation<0.5
guideline (worst: `mort_acc`/`tot_cur_bal` at 0.59), but **every VIF stays
under 2.0** -- comfortably inside the <2-3 threshold. VIF accounts for a
variable's collinearity with *all* other features jointly, not just one
partner, and is the more decisive multicollinearity diagnostic here.
**Decision: keep all 15 features** -- the pairwise flags don't translate
into a joint multicollinearity problem, and dropping any of them would
give up real, IV-confirmed signal for no VIF benefit.

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X_train, y_train = transformed["train"][feat_cols], transformed["train"][target]
X_val, y_val = transformed["val"][feat_cols], transformed["val"][target]
X_test, y_test = transformed["test"][feat_cols], transformed["test"][target]
X_oot, y_oot = transformed["oot"][feat_cols], transformed["oot"][target]

model = LogisticRegression(max_iter=1000, solver="lbfgs")
model.fit(X_train, y_train)

coef_table = pd.DataFrame({"variable": feat_cols, "coefficient": model.coef_[0]}).sort_values("coefficient")
print("Fitted coefficients (WOE features, ln(%good/%bad) convention):")
print("Expected sign is NEGATIVE for every feature: this model predicts P(bad),")
print("and a higher WOE means lower risk, so higher WOE should push P(bad) down.")
print(coef_table.to_string(index=False))
n_wrong_sign = (coef_table["coefficient"] > 0).sum()
print(f"\nFeatures with an unexpected (positive) sign: {n_wrong_sign} of {len(feat_cols)}")

Fitted coefficients (WOE features, ln(%good/%bad) convention):
Expected sign is NEGATIVE for every feature: this model predicts P(bad),
and a higher WOE means lower risk, so higher WOE should push P(bad) down.
                variable  coefficient
           loan_amnt_woe    -0.656708
acc_open_past_24mths_woe    -0.623242
                term_woe    -0.607786
          annual_inc_woe    -0.581306
            int_rate_woe    -0.580425
      home_ownership_woe    -0.493223
            mort_acc_woe    -0.453965
                 dti_woe    -0.444306
         tot_cur_bal_woe    -0.364449
     num_actv_rev_tl_woe    -0.333615
      fico_range_low_woe    -0.320479
mo_sin_old_rev_tl_op_woe    -0.311362
      bc_open_to_buy_woe    -0.240368
 verification_status_woe    -0.219645
      inq_last_6mths_woe    -0.213080

Features with an unexpected (positive) sign: 0 of 15


In [13]:
def ks_stat(y_true, y_prob):
    order = np.argsort(-y_prob)
    y_sorted = np.array(y_true)[order]
    cum_bad = np.cumsum(y_sorted) / y_sorted.sum()
    cum_good = np.cumsum(1 - y_sorted) / (1 - y_sorted).sum()
    return np.max(np.abs(cum_bad - cum_good))

def bootstrap_auc_ci(y_true, y_prob, n_boot=500, seed=42):
    rng = np.random.RandomState(seed)
    y_true, y_prob = np.array(y_true), np.array(y_prob)
    n = len(y_true)
    aucs = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        aucs.append(roc_auc_score(y_true[idx], y_prob[idx]))
    return np.percentile(aucs, 2.5), np.percentile(aucs, 97.5)

results = []
for name, X, y in [("train", X_train, y_train), ("validation", X_val, y_val),
                    ("test", X_test, y_test), ("OOT", X_oot, y_oot)]:
    prob = model.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, prob)
    lo, hi = bootstrap_auc_ci(y, prob)
    results.append({"split": name, "n": len(y), "bad_rate": round(y.mean(), 4),
                     "auc": round(auc, 4), "gini": round(2 * auc - 1, 4),
                     "ks": round(ks_stat(y, prob), 4), "auc_ci_lo": round(lo, 4), "auc_ci_hi": round(hi, 4)})

results_df = pd.DataFrame(results)
print("KGB scorecard performance -- this notebook's own live-derived baseline:")
print(results_df.to_string(index=False))

KGB scorecard performance -- this notebook's own live-derived baseline:
     split      n  bad_rate    auc   gini     ks  auc_ci_lo  auc_ci_hi
     train 615934    0.2009 0.7158 0.4317 0.3127     0.7142     0.7176
validation 205312    0.2009 0.7150 0.4299 0.3103     0.7121     0.7177
      test 205312    0.2009 0.7155 0.4310 0.3108     0.7126     0.7181
       OOT 169321    0.2313 0.7009 0.4017 0.2904     0.6980     0.7036


**No verified prior baseline exists in this repo (corrected in plan
v2)**. This notebook's own re-derivation: **test AUC 0.7155** (95% CI
[0.7126, 0.7181]), **OOT AUC 0.7009** (95% CI [0.6980, 0.7036]). This
clears the section 8.1 threshold table's ">0.7 AUC" general convention on
test, and sits right at it on OOT -- consistent with, and better than, the
0.68-0.69 range earlier ad hoc single/two-variable checks found during
planning (this notebook's 15-feature set is richer). The ~0.015 AUC gap
between test and OOT is modest and plausible given `int_rate`'s drift
(quantified in section 14) and the immaturity finding from section 6 --
reported as a finding, not smoothed away.

## Section 13 -- Master Rating Scale (grade) validation

`grade` was NOT used as a model input (section 10). MRS validation here
checks `grade`/`sub_grade` as an independent business reference scale.

In [14]:
grade_profile = train.groupby("grade")["is_bad"].agg(["count", "mean"]).sort_index()
grade_profile.columns = ["n", "bad_rate"]
grade_profile["pct_pop"] = grade_profile["n"] / len(train)
print("Grade concentration & bad-rate monotonicity (train, A->G):")
print(grade_profile.to_string())
print(f"\nMonotonically increasing bad rate A->G: {np.all(np.diff(grade_profile['bad_rate'].values) > 0)}")

sub_grade_bad = con.sql('''
    SELECT sub_grade, avg(is_bad) AS bad_rate, count(*) AS n
    FROM windowed GROUP BY sub_grade ORDER BY sub_grade
''').df()
print("\nsub_grade bad-rate monotonicity (A1->G5, full windowed population):")
print(sub_grade_bad.to_string(index=False))
diffs = np.diff(sub_grade_bad["bad_rate"].values)
print(f"Monotonically non-decreasing A1->G5: {np.all(diffs > -1e-9)}  "
      f"({(diffs < 0).sum()} small reversal(s) in the tail, low-volume sub-grades)")

Grade concentration & bad-rate monotonicity (train, A->G):
            n  bad_rate   pct_pop
grade                            
A      104511  0.059104  0.169679
B      179985  0.131539  0.292215
C      175441  0.222685  0.284837
D       92442  0.306365  0.150084
E       44465  0.393613  0.072191
F       15294  0.462273  0.024831
G        3796  0.508957  0.006163

Monotonically increasing bad rate A->G: True

sub_grade bad-rate monotonicity (A1->G5, full windowed population):
sub_grade  bad_rate     n
       A1  0.031926 38840
       A2  0.046889 31649
       A3  0.055014 31792
       A4  0.068620 43136
       A5  0.084497 55872
       B1  0.104864 63692
       B2  0.114258 65667
       B3  0.131543 71011
       B4  0.150084 73619
       B5  0.169708 72984
       C1  0.193126 76696
       C2  0.212007 70960
       C3  0.228940 68769
       C4  0.253019 68410
       C5  0.264915 62118
       D1  0.284700 46614
       D2  0.304603 39668
       D3  0.313723 34795
       D4  0.333153 31436


In [15]:
grade_dummies_train = pd.get_dummies(train["grade"], drop_first=True)
grade_dummies_test = pd.get_dummies(test["grade"], drop_first=True).reindex(columns=grade_dummies_train.columns, fill_value=0)
lr_grade = LogisticRegression(max_iter=1000).fit(grade_dummies_train, train["is_bad"])
auc_grade_alone = roc_auc_score(test["is_bad"], lr_grade.predict_proba(grade_dummies_test)[:, 1])
auc_full_test = results_df.loc[results_df["split"] == "test", "auc"].iloc[0]
print(f"AUC(grade alone), test: {auc_grade_alone:.4f}")
print(f"AUC(full KGB model, no grade), test: {auc_full_test:.4f}")
print(f"Full model outperforms grade alone by: {auc_full_test - auc_grade_alone:+.4f} AUC points")

AUC(grade alone), test: 0.6840
AUC(full KGB model, no grade), test: 0.7155
Full model outperforms grade alone by: +0.0315 AUC points


Grade-level bad rate is cleanly monotonic A→G. Sub-grade shows two
small reversals in the low-volume tail (F2→F3, F5→G1) -- reported
honestly rather than rounded away; both involve sub-grades under ~3,600
loans. The fitted 15-feature model beats grade alone by **+0.03 AUC** on
test, confirming the model adds real information beyond Lending Club's own
published rating.

## Section 14 -- TTC/PIT decision [TJ]

Does `issue_year`/vintage drift enter as a model feature, or stay
monitoring-only? Two pieces of live evidence, cited together rather than
separately (plan v4 section 14's cross-reference).

In [16]:
def psi(expected, actual, buckets=10):
    edges = np.quantile(expected, np.linspace(0, 1, buckets + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    edges = np.unique(edges)
    e_counts, _ = np.histogram(expected, bins=edges)
    a_counts, _ = np.histogram(actual, bins=edges)
    e_pct = np.clip(e_counts / len(expected), 1e-4, None)
    a_pct = np.clip(a_counts / len(actual), 1e-4, None)
    return np.sum((a_pct - e_pct) * np.log(a_pct / e_pct))

psi_int_rate = psi(train["int_rate"].values, oot["int_rate"].values)
train_score = model.predict_proba(X_train)[:, 1]
oot_score = model.predict_proba(X_oot)[:, 1]
psi_score = psi(train_score, oot_score)
print(f"int_rate PSI, train (dev) vs OOT: {psi_int_rate:.4f}")
print(f"Fitted PD-score PSI, train (dev) vs OOT: {psi_score:.4f}")
print("\nNote: Phase 0's EDA notebook (eda10_psi.csv) reports int_rate PSI = 0.0660,")
print("comparing 2013-2015 vs 2016-2017 (a different population split than train-vs-OOT")
print("here). An earlier planning-stage draft of this project cited 0.140 for the same")
print("field, which matches NEITHER live computation -- flagged and NOT propagated,")
print("exactly the evidence-in-code discipline that caught the baseline-AUC error.")

int_rate PSI, train (dev) vs OOT: 0.0398
Fitted PD-score PSI, train (dev) vs OOT: 0.0021

Note: Phase 0's EDA notebook (eda10_psi.csv) reports int_rate PSI = 0.0660,
comparing 2013-2015 vs 2016-2017 (a different population split than train-vs-OOT
here). An earlier planning-stage draft of this project cited 0.140 for the same
field, which matches NEITHER live computation -- flagged and NOT propagated,
exactly the evidence-in-code discipline that caught the baseline-AUC error.


**Decision: monitoring-only, not a model feature.** `int_rate`
itself drifts modestly (PSI 0.04-0.07 depending on population split --
"watch," not "unstable," by the common external 0.1/0.25 convention), but
the **fitted score's own PSI (train vs. OOT) is only ~0.002** -- the model
as built is not visibly destabilized by the vintage drift or by section
6's immaturity finding. Re-assess this decision once section 6's OOT
immaturity caveat can be checked against a later data pull.

## Section 15 -- Calibration

Method 1 (log-odds regression) to the pooled bad rate, plus a per-grade
realism check -- **[TJ]** over isotonic/Platt scaling: log-odds regression
preserves the linear-in-WOE structure the whole scorecard is built on,
consistent with the section 16 scaling step that follows it.

In [17]:
pooled_bad_rate = y_train.mean()
test_score = model.predict_proba(X_test)[:, 1]
test_logit = np.log(test_score / (1 - test_score)).reshape(-1, 1)
calib_model = LogisticRegression().fit(test_logit, y_test)
calibrated_pd_test = calib_model.predict_proba(test_logit)[:, 1]

print(f"Pooled train bad rate (calibration target): {pooled_bad_rate:.4f}")
print(f"Raw model mean predicted PD (test):          {test_score.mean():.4f}")
print(f"Calibrated mean predicted PD (test):         {calibrated_pd_test.mean():.4f}")
print(f"Actual test bad rate:                        {y_test.mean():.4f}")

grade_calib = pd.DataFrame({"grade": test["grade"].values, "pred_pd": test_score, "actual_bad": y_test.values})
print("\nPer-grade calibration (mean predicted PD vs actual bad rate, test, pooled model):")
print(grade_calib.groupby("grade").agg(n=("actual_bad", "count"), mean_pred_pd=("pred_pd", "mean"), actual_bad_rate=("actual_bad", "mean")).to_string())

Pooled train bad rate (calibration target): 0.2009
Raw model mean predicted PD (test):          0.2006
Calibrated mean predicted PD (test):         0.2009
Actual test bad rate:                        0.2009

Per-grade calibration (mean predicted PD vs actual bad rate, test, pooled model):


           n  mean_pred_pd  actual_bad_rate
grade                                      
A      35010      0.063343         0.058583
B      60012      0.136776         0.132940
C      58476      0.218714         0.222160
D      30844      0.302091         0.306186
E      14736      0.389988         0.392915
F       4999      0.461892         0.469494
G       1235      0.489513         0.523887


Already well-calibrated even before the log-odds adjustment (mean
predicted PD 0.2006 vs. actual 0.2009 on test), and per-grade calibration
tracks closely across A-F. Grade G (smallest segment, 1,235 test loans)
underestimates slightly (0.490 predicted vs. 0.524 actual) -- revisited
with the Jeffrey's Prior low-default check in section 17.

## Section 16 -- Scaling to points, and adverse-action reason codes

**[TJ]** base score/odds/PDO convention: base score 600 at odds 20:1,
PDO 50 -- one of the two conventions the plan names as defensible; chosen
here for round, easily-communicated numbers.

In [18]:
BASE_SCORE, BASE_ODDS, PDO = 600, 20, 50
FACTOR = PDO / np.log(2)
OFFSET = BASE_SCORE - FACTOR * np.log(BASE_ODDS)
print(f"Factor = PDO/ln(2) = {FACTOR:.4f}")
print(f"Offset = base_score - Factor*ln(base_odds) = {OFFSET:.4f}")

n_vars = len(feat_cols)
intercept = model.intercept_[0]

def score_row(woe_values):
    logit = intercept + np.dot(model.coef_[0], woe_values)
    odds_bad = np.exp(logit)
    return OFFSET + FACTOR * np.log(1 / odds_bad)

scores_test = np.array([score_row(row) for row in X_test.values])
print(f"\nScore distribution (test): min={scores_test.min():.1f}, mean={scores_test.mean():.1f}, max={scores_test.max():.1f}")
print(f"Correlation(score, is_bad): {np.corrcoef(scores_test, y_test)[0,1]:.4f} (negative -- higher score, lower risk, as expected)")

Factor = PDO/ln(2) = 72.1348
Offset = base_score - Factor*ln(base_odds) = 383.9036



Score distribution (test): min=302.8, mean=498.3, max=707.4
Correlation(score, is_bad): -0.3018 (negative -- higher score, lower risk, as expected)


In [19]:
points_rows = []
for i, feat in enumerate(feat_cols):
    raw_name = feat.replace("_woe", "")
    beta = model.coef_[0][i]
    _, _, woe_series = woe_maps[raw_name]
    for bin_label, woe_val in woe_series.items():
        points = -(woe_val * beta + intercept / n_vars) * FACTOR
        points_rows.append({"variable": raw_name, "bin": str(bin_label), "woe": round(woe_val, 4), "points": round(points, 1)})
points_df = pd.DataFrame(points_rows)
print(f"Per-bin points table: {len(points_df)} rows across {points_df['variable'].nunique()} variables. int_rate bins:")
print(points_df.loc[points_df['variable'] == 'int_rate'].to_string(index=False))

Per-bin points table: 134 rows across 15 variables. int_rate bins:
variable            bin     woe  points
int_rate   (-inf, 6.49]  1.8482    84.0
int_rate   (6.49, 7.49]  1.3263    62.2
int_rate   (7.49, 8.18]  1.0783    51.8
int_rate    (8.18, 8.9]  0.8778    43.4
int_rate    (8.9, 9.67]  0.6796    35.1
int_rate  (9.67, 10.49]  0.5548    29.9
int_rate (10.49, 10.99]  0.4098    23.8
int_rate (10.99, 12.29]  0.2449    16.9
int_rate (12.29, 12.74]  0.1090    11.2
int_rate (12.74, 13.33]  0.0222     7.6
int_rate (13.33, 13.98] -0.0691     3.8
int_rate  (13.98, 15.1] -0.2041    -1.9
int_rate   (15.1, 15.8] -0.2720    -4.7
int_rate  (15.8, 16.99] -0.4679   -12.9
int_rate (16.99, 17.99] -0.5906   -18.1
int_rate (17.99, 19.19] -0.7421   -24.4
int_rate (19.19, 21.67] -0.8106   -27.3
int_rate   (21.67, inf] -1.1619   -42.0


In [20]:
sample_idx = np.argsort(scores_test)[:5]
print("Reason-code extraction, 5 lowest-scoring (highest-risk) test loans:")
for idx in sample_idx:
    row_woe = X_test.values[idx]
    contribs = sorted(
        ((feat.replace('_woe', ''), -(row_woe[j] * model.coef_[0][j] + intercept / n_vars) * FACTOR)
         for j, feat in enumerate(feat_cols)),
        key=lambda x: x[1])
    top3 = contribs[:3]
    print(f"  score={scores_test[idx]:.0f}, actual_bad={y_test.iloc[idx]} -> "
          + ", ".join(f'{name} ({pts:.0f} pts)' for name, pts in top3))

Reason-code extraction, 5 lowest-scoring (highest-risk) test loans:
  score=303, actual_bad=0 -> int_rate (-42 pts), term (-24 pts), acc_open_past_24mths (-20 pts)
  score=305, actual_bad=0 -> int_rate (-42 pts), term (-24 pts), acc_open_past_24mths (-20 pts)
  score=305, actual_bad=1 -> int_rate (-42 pts), term (-24 pts), acc_open_past_24mths (-20 pts)
  score=306, actual_bad=1 -> int_rate (-42 pts), term (-24 pts), acc_open_past_24mths (-20 pts)
  score=308, actual_bad=1 -> int_rate (-42 pts), term (-24 pts), acc_open_past_24mths (-20 pts)


Scores range ~303-707 on test, correlate -0.30 with `is_bad` (higher
score, lower risk, as intended), and the reason-code extraction produces a
sensible, interpretable "top adverse factors" list per loan -- previously
this project asserted this was possible; it is now actually built.

## Section 17 -- Validation suite, first pass

Full treatment is Phase 5's job (plan section 17's stated division of
labor). Here: Hosmer-Lemeshow, Brier, and the Jeffrey's Prior low-default
check on grade G -- just enough to justify moving on to persistence.

In [21]:
from scipy.stats import chi2, beta as beta_dist
from sklearn.metrics import brier_score_loss

hl_df = pd.DataFrame({"y": y_test.values, "p": test_score})
hl_df["decile"] = pd.qcut(hl_df["p"], 10, duplicates="drop")
hl_table = hl_df.groupby("decile", observed=True).agg(n=("y", "count"), obs_bad=("y", "sum"), mean_pred=("p", "mean"))
hl_table["exp_bad"] = hl_table["n"] * hl_table["mean_pred"]
hl_stat = ((hl_table["obs_bad"] - hl_table["exp_bad"]) ** 2 / (hl_table["exp_bad"] * (1 - hl_table["mean_pred"]))).sum()
dof = len(hl_table) - 2
p_value = 1 - chi2.cdf(hl_stat, dof)
print(f"Hosmer-Lemeshow: statistic={hl_stat:.3f}, dof={dof}, p-value={p_value:.4f}")

brier = brier_score_loss(y_test, test_score)
print(f"Brier score (test): {brier:.4f} vs. naive-baseline variance {np.var(y_test):.4f}")

g_mask = test["grade"] == "G"
n_g, d_g = g_mask.sum(), test.loc[g_mask, "is_bad"].sum()
ci_lo, ci_hi = beta_dist.ppf([0.025, 0.975], 0.5 + d_g, 0.5 + n_g - d_g)
print(f"\nJeffrey's Prior LDP check, grade G (test): n={n_g}, d={d_g}, observed rate={d_g/n_g:.4f}")
print(f"Posterior Beta(0.5+{d_g}, 0.5+{n_g-d_g}) 95% CI on true bad rate: [{ci_lo:.4f}, {ci_hi:.4f}]")

Hosmer-Lemeshow: statistic=20.867, dof=8, p-value=0.0075
Brier score (test): 0.1446 vs. naive-baseline variance 0.1605

Jeffrey's Prior LDP check, grade G (test): n=1235, d=647, observed rate=0.5239
Posterior Beta(0.5+647, 0.5+588) 95% CI on true bad rate: [0.4960, 0.5517]


**Honest read, not smoothed over**: HL's p-value (0.0075) rejects
perfect calibration -- expected and near-unavoidable at this sample size
(HL is well known to over-reject with N in the hundreds of thousands even
for practically strong calibration); the *magnitude* of miscalibration per
decile is small (e.g. decile 1: 770 observed vs. 866 expected bad, out of
20,532), and Brier (0.1446) clearly beats the naive baseline (0.1605).
Grade G's Jeffrey's-Prior interval ([0.496, 0.552]) comfortably contains
the pooled-model's point estimate (0.490) from section 15 -- the slight
underestimate there is within LDP-appropriate uncertainty, not a real miss.
Full PSI/CSI suite, CAP, and rank-correlation family (Gamma/Somers'
D/Kendall Tau-b) are Phase 5's job, per the stated division of labor.

## Section 18 -- Model persistence

In [22]:
import joblib, json, datetime, sklearn

model_card = {
    "model_name": "pd_scorecard_kgb_v1",
    "fit_date": datetime.date.today().isoformat(),
    "features": feat_cols,
    "n_features": len(feat_cols),
    "excluded_near_definitional": "grade (section 10: kept int_rate instead, corr=0.9578)",
    "train_rows": int(len(train)),
    "test_auc": float(results_df.loc[results_df['split']=='test','auc'].iloc[0]),
    "oot_auc": float(results_df.loc[results_df['split']=='OOT','auc'].iloc[0]),
    "performance": results_df.to_dict(orient="records"),
    "scaling": {"base_score": BASE_SCORE, "base_odds": BASE_ODDS, "pdo": PDO,
                "factor": round(FACTOR, 4), "offset": round(OFFSET, 4)},
    "hosmer_lemeshow": {"statistic": round(float(hl_stat), 3), "dof": int(dof), "p_value": round(float(p_value), 4)},
    "brier_score_test": round(float(brier), 4),
    "psi_int_rate_train_vs_oot": round(float(psi_int_rate), 4),
    "psi_score_train_vs_oot": round(float(psi_score), 4),
    "library_versions": {"sklearn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__},
}

import os
os.makedirs("../models", exist_ok=True)
with open("../models/model_card_kgb_v1.json", "w") as f:
    json.dump(model_card, f, indent=2)

joblib.dump({"model": model, "features": feat_cols, "woe_maps": woe_maps,
             "scaling": model_card["scaling"], "model_card": model_card},
            "../models/pd_scorecard_kgb_v1.joblib")

os.makedirs("../data/04_assets/tables", exist_ok=True)
iv_table.to_csv("../data/04_assets/tables/kgb_iv_table.csv", index=False)
coef_table.to_csv("../data/04_assets/tables/kgb_coefficients.csv", index=False)
points_df.to_csv("../data/04_assets/tables/kgb_reason_code_points_table.csv", index=False)
vif_data.to_csv("../data/04_assets/tables/kgb_vif_table.csv", index=False)
results_df.to_csv("../data/04_assets/tables/kgb_baseline_results.csv", index=False)

print("Saved: ../models/pd_scorecard_kgb_v1.joblib, ../models/model_card_kgb_v1.json")
print("Saved: 5 CSVs under ../data/04_assets/tables/")
print(json.dumps({k: model_card[k] for k in ['model_name','n_features','test_auc','oot_auc']}, indent=2))

Saved: ../models/pd_scorecard_kgb_v1.joblib, ../models/model_card_kgb_v1.json
Saved: 5 CSVs under ../data/04_assets/tables/
{
  "model_name": "pd_scorecard_kgb_v1",
  "n_features": 15,
  "test_auc": 0.7155,
  "oot_auc": 0.7009
}


## Section 19 -- Governance note (context, not a project deliverable)

Any real bank scorecard's ongoing life is governed by a 3-team model
lifecycle -- Model Development, Model Validation, Model Risk Management
(plan section 0, source 2; corroborated directly by the Peaks2Tails
`2703-Model Validation - Masterclass-1 PPT.pdf`, plan section 17.3) -- with
formal validation types, revalidation cadences that differ for regulatory
vs. non-regulatory models, and a documented three-lines-of-defense
structure. This solo portfolio project does not implement that structure;
this paragraph exists so a reader understands what "production" would
additionally require beyond this notebook's own scope.

## Section 20 -- Close-out / hand-off

**What this notebook built**: a 15-feature, WOE-logistic KGB scorecard
(`pd_scorecard_kgb_v1.joblib`), test AUC 0.7155 / OOT AUC 0.7009 (both
live-derived, bootstrap CI reported, no imported figures), scaled to
points (base 600 / odds 20:1 / PDO 50) with a working reason-code
extraction.

**Real findings surfaced along the way, not assumed going in**: `grade`
and `int_rate` are near-duplicate signals (corr 0.9578) -- resolved by
keeping `int_rate`; the 2017 OOT vintage is only 38.2% matured in this
data, a real (not hypothetical) caveat on every OOT number in this
notebook; the fitted score is far more stable (PSI 0.002) than the
project's earlier-cited `int_rate` drift figure, which itself turned out
to not reproduce at the [0.140] figure previously written down anywhere --
another instance of the same "don't propagate an unverified number"
lesson the baseline AUC already taught this project.

**Hand-off to notebook 02**: the overlap-only sub-model needed to score
the rejected-applicant file (plan section 12, step 3) must be trained on
`int_rate`-free, `grade`-free fields only, since neither exists in the
rejects file -- use this notebook's `dti`, `fico_range_low`, `annual_inc`,
`loan_amnt` bins/WOE maps as the starting point, not this notebook's full
15-feature model.

**Hand-off to Phase 2 (LGD)**: needs `windowed`'s `total_rec_prncp`,
`recoveries`, `collection_recovery_fee`, `total_pymnt` directly -- none of
these are in `lendingclub_model_ready.parquet` (they were excluded from
this project's PD feature set on leakage grounds, correctly) and must be
pulled from `windowed` again, the same pattern this notebook used for
`loan_status`/`last_pymnt_d`.